In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

# --- TUA FUNZIONE METRICA UFFICIALE ---
def get_score(y_true, y_pred, component_type):
    """Calcola il punteggio ufficiale penalizzando i ritardi."""
    error = y_pred - y_true
    alpha = 0.01
    max_val = np.max(y_true) if len(y_true) > 0 else 1
    # Beta varia in base al tipo di componente
    beta = 1/max_val if component_type == 'WW' else 2/max_val
    # Penalità doppia per errori positivi (ritardo)
    w = np.where(error >= 0, 2/(1+alpha*y_true), 1/(1+alpha*y_true))
    return np.mean(w * (error**2) * beta)

# --- CONFIGURAZIONE ---
PATH_INPUT = 'data_elaborated/train/train_with_residuals.csv'
DIR_MODELS = 'models/'
TARGETS = ['Cycles_to_WW', 'Cycles_to_HPC_SV', 'Cycles_to_HPT_SV']
SAFETY_MARGIN = 0.90 

def train_with_official_score():
    if not os.path.exists(PATH_INPUT):
        print(f"File {PATH_INPUT} non trovato!")
        return

    df = pd.read_csv(PATH_INPUT)
    
    # Feature selection (Sensori + Residui)
    features = [c for c in df.columns if 'Sensed' in c or '_res' in c]
    features = [f for f in features if f not in ['ESN', 'Cycles_Since_New', 'Snapshot'] + TARGETS]

    for target in TARGETS:
        print(f"\n>>> Training e Valutazione per: {target}")
        
        # Pulizia e Split
        current_df = df.dropna(subset=[target])
        X = current_df[features]
        y = current_df[target]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # Modello veloce
        model = HistGradientBoostingRegressor(max_iter=100, max_depth=7, random_state=42)
        model.fit(X_train, y_train)

        # Predizione con margine di sicurezza
        y_pred_safe = model.predict(X_test) * SAFETY_MARGIN
        
        # --- CALCOLO SCORE UFFICIALE ---
        # Estraiamo il tipo di componente dal nome del target (WW, HPC, o HPT)
        comp_type = 'WW' if 'WW' in target else 'SV'
        official_score = get_score(y_test.values, y_pred_safe, comp_type)
        mae = mean_absolute_error(y_test, y_pred_safe)

        print(f"MAE: {mae:.2f} cicli")
        print(f"OFFICIAL PHM SCORE: {official_score:.4f}")
        
        # Salvataggio
        if not os.path.exists(DIR_MODELS): os.makedirs(DIR_MODELS)
        joblib.dump(model, os.path.join(DIR_MODELS, f"model_{target}.pkl"))

if __name__ == "__main__":
    train_with_official_score()


>>> Training e Valutazione per: Cycles_to_WW
MAE: 230.88 cicli
OFFICIAL PHM SCORE: 29.7839

>>> Training e Valutazione per: Cycles_to_HPC_SV
MAE: 967.57 cicli
OFFICIAL PHM SCORE: 49.9452

>>> Training e Valutazione per: Cycles_to_HPT_SV
MAE: 276.49 cicli
OFFICIAL PHM SCORE: 10.0018
